# Proyecto Integrador 2 · Parte VI — Representación en grafos
## Vectores para cada proveedor: SVD y node2vec sobre el grafo de co-participación

**Qué hace este notebook**
1. Toma un **núcleo** del grafo de co-participación (los proveedores más activos).
2. Convierte cada proveedor en un **vector de 16 números** (*embedding*) con dos técnicas de clase: **SVD** y **node2vec**.
3. Usa los vectores en dos tareas: **agrupar** (K-Means) y **visualizar** (t-SNE).
4. Mide **qué estructura capturan** y cómo se relaciona con las **comunidades** del notebook 8 y con los **clústeres por características** de la Parte II.
5. Prueba cómo cambian los resultados con los parámetros **p** y **q** de node2vec.

**Dónde corre cada cosa:** todo **en local** (NumPy, scikit-learn, gensim). El núcleo tiene unos 3,500 proveedores, cabe en memoria y no necesita Spark (la Parte VI lo deja como opcional).

**Entrada:** `grafo_aristas`, `features_proveedor` (notebooks 1 y 2), `comunidades_louvain` y `perfil_proveedor` (notebook 8). Para compararlos con los tipos de proveedor de la Parte II se repite ese mismo K-Means sobre `features_proveedor`.
**Salida:** archivos `emb_*` para la página web.

**Qué se vio en clase**
- SVD como embebedor, node2vec (caminatas sesgadas con p y q, muestreo negativo), K-Means, t-SNE, ARI: **vistos en clase**.
- **Nota de implementación:** las caminatas sesgadas de node2vec (con p y q) se programan aquí en unas 20 líneas en vez de instalar el paquete `node2vec`, que no soporta bien los nodos con miles de vecinos; el entrenamiento skip-gram con muestreo negativo usa `gensim`, igual que el paquete. Es el mismo algoritmo de clase: no se usa ninguna técnica fuera de clase.
- **No se usa GNN:** el enunciado pide «al menos una técnica» y aquí hay dos.

## 0. Configuración

In [1]:
import os, json, time
import numpy as np
import pandas as pd
import networkx as nx

USAR_DRIVE = False   # True si corres en Colab con Drive
RUTA_P2 = os.environ.get("P2_DATOS", "/content/drive/MyDrive/proyecto_integrador_seace/p2")
WEB     = os.environ.get("P2_WEB",   "/content/drive/MyDrive/proyecto_integrador_seace/web_artefactos/p2")

MIN_PART_NUCLEO = 120    # el núcleo son los proveedores con al menos 120 participaciones
D        = 16            # tamaño del embedding
NUM_WALKS, LARGO, VENTANA = 10, 40, 5     # caminatas por nodo, largo de cada una, ventana del skip-gram
GRILLA_P = [0.5, 1.0, 2.0]
GRILLA_Q = [0.5, 1.0, 2.0]
P_BASE, Q_BASE = 1.0, 0.5   # valores del ejemplo de clase (se prefiere explorar comunidades: q bajo)
MIN_GRANDE = 100         # una comunidad cuenta como «grande» si tiene al menos 100 proveedores en el núcleo
SEMILLA  = 42

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(WEB, exist_ok=True)

def guardar_json(nombre, obj):
    with open(f"{WEB}/{nombre}", "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    print("guardado:", nombre)

def guardar_csv(nombre, df):
    df.to_csv(f"{WEB}/{nombre}", index=False)
    print("guardado:", nombre, len(df), "filas")

## 1. El núcleo del grafo
**Por qué un núcleo.** Los embeddings aprenden de **cada nodo y de sus vecinos**. Un proveedor con pocas conexiones da casi nada que aprender, y node2vec debe simular caminatas desde cada nodo. Se toman los **proveedores con al menos 120 participaciones** y las aristas con **peso ≥ 2** entre ellos (las del grafo del notebook 2), y se queda la componente conexa más grande.

**Limitación:** los resultados describen a los **proveedores más activos**, no a todo el universo.

In [2]:
aristas = pd.read_parquet(f"{RUTA_P2}/grafo_aristas")
feat = pd.read_parquet(f"{RUTA_P2}/features_proveedor", columns=["proveedor_idx", "ruc", "nombre", "n_participaciones"]).set_index("proveedor_idx")
nucleo = set(feat[feat.n_participaciones >= MIN_PART_NUCLEO].index)
e = aristas[aristas.prov_a.isin(nucleo) & aristas.prov_b.isin(nucleo)]
G = nx.Graph(); G.add_weighted_edges_from(e[["prov_a", "prov_b", "peso"]].itertuples(index=False, name=None))
G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
nodos = sorted(G.nodes()); pos = {n: i for i, n in enumerate(nodos)}
n = len(nodos)
grado = np.array([G.degree(x) for x in nodos])
print(f"núcleo: {n:,} proveedores, {G.number_of_edges():,} aristas, grado medio {grado.mean():.0f}, densidad {nx.density(G):.3f}")

# Etiquetas de otras partes del proyecto
com = pd.read_parquet(f"{RUTA_P2}/comunidades_louvain").set_index("proveedor_idx").comunidad
perfil = pd.read_parquet(f"{RUTA_P2}/perfil_proveedor").set_index("proveedor_idx")
etq_com = com.reindex(nodos).values
tam_com = pd.Series(etq_com).value_counts()
grandes = tam_com[tam_com >= MIN_GRANDE].index.tolist()
K = len(grandes)
en_grande = np.isin(etq_com, grandes)
print(f"comunidades de Louvain presentes en el núcleo: {tam_com.size} | grandes (≥ {MIN_GRANDE}): {K}, con {en_grande.sum():,} proveedores")

núcleo: 3,489 proveedores, 328,297 aristas, grado medio 188, densidad 0.054
comunidades de Louvain presentes en el núcleo: 18 | grandes (≥ 100): 12, con 3,219 proveedores


## 2. Embedding A — SVD (clase: «SVD como embebedor»)
**Idea de clase:** la SVD descompone una matriz en `A ≈ U·Σ·Vᵀ`; las filas de `U·Σ` son un **vector corto por fila**. Aquí la matriz es la **adyacencia ponderada** del núcleo (`log(1 + peso)`, para que ni un par con peso enorme domine). Cada proveedor queda representado por los `D = 16` «conceptos» más fuertes de la red.

Se usa `TruncatedSVD` de scikit-learn, que es la SVD para matrices dispersas (calcula solo los primeros `D` valores singulares). **Todos los embeddings se normalizan** (largo 1) para comparar por ángulo.

In [3]:
from scipy.sparse import coo_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

filas = [pos[a] for a, b in G.edges()] + [pos[b] for a, b in G.edges()]
cols  = [pos[b] for a, b in G.edges()] + [pos[a] for a, b in G.edges()]
vals  = [np.log1p(G[a][b]["weight"]) for a, b in G.edges()] * 2
Aw = coo_matrix((vals, (filas, cols)), shape=(n, n)).tocsr()

svd = TruncatedSVD(n_components=D, random_state=SEMILLA)
Z_svd = normalize(svd.fit_transform(Aw))
print("varianza explicada por los 16 componentes:", round(float(svd.explained_variance_ratio_.sum()), 3), "| primeros 3:", svd.explained_variance_ratio_[:3].round(3))

varianza explicada por los 16 componentes: 0.553 | primeros 3: [0.117 0.121 0.061]


## 3. Embedding B — node2vec (clase)
**Idea de clase (Word2Vec en grafos):** en una frase, las palabras que aparecen cerca tienen vectores parecidos. En un grafo se **simulan caminatas aleatorias**: cada caminata es una «frase» y los nodos que aparecen juntos en caminatas tendrán vectores parecidos.

**node2vec** usa caminatas **sesgadas** con dos parámetros:
| Parámetro | Valor | Efecto |
|---|---|---|
| **p** | alto | evita volver al nodo anterior: **explora más** |
| **p** | bajo | vuelve seguido: **se queda cerca** del origen |
| **q** | alto | se queda cerca del nodo anterior: tipo **BFS** (vecindario local, roles) |
| **q** | bajo | se aleja: tipo **DFS** (vecindario amplio, comunidades) |

Con `p = q = 1` es la caminata uniforme (DeepWalk). Luego se entrena un **skip-gram con muestreo negativo** sobre las caminatas.

**Pesos:** al elegir el siguiente nodo, la probabilidad es proporcional al **peso** de la arista (procesos en común) multiplicado por el sesgo `α(p, q)`. Para que sea rápido con nodos de miles de vecinos se usa **muestreo con rechazo**: se propone un vecino según el peso y se acepta con probabilidad `α / α_máx`.

In [4]:
from gensim.models import Word2Vec

vecinos = [np.array([pos[x] for x in G.neighbors(v)]) for v in nodos]
pesos   = [np.array([G[v][x]["weight"] for x in G.neighbors(v)], dtype=float) for v in nodos]
acum    = [np.cumsum(w) for w in pesos]
conj    = [set(v.tolist()) for v in vecinos]

def caminatas(p, q, semilla=SEMILLA):
    rng = np.random.default_rng(semilla)
    a_max = max(1 / p, 1.0, 1 / q)
    res = []
    for _ in range(NUM_WALKS):
        for inicio in rng.permutation(n):
            cam = [int(inicio)]
            for _ in range(LARGO - 1):
                v = cam[-1]
                t = cam[-2] if len(cam) > 1 else -1
                while True:
                    j = int(np.searchsorted(acum[v], rng.random() * acum[v][-1]))
                    x = int(vecinos[v][min(j, len(vecinos[v]) - 1)])
                    if t < 0:
                        break
                    alfa = 1 / p if x == t else (1.0 if x in conj[t] else 1 / q)
                    if rng.random() < alfa / a_max:
                        break
                cam.append(x)
            res.append([str(c) for c in cam])
    return res

def node2vec(p, q, semilla=SEMILLA):
    t0 = time.time()
    cams = caminatas(p, q, semilla)
    modelo = Word2Vec(cams, vector_size=D, window=VENTANA, min_count=1, sg=1, negative=5, workers=1, epochs=5, seed=semilla)
    Z = np.vstack([modelo.wv[str(i)] for i in range(n)])
    return normalize(Z), time.time() - t0

Z_base, seg = node2vec(P_BASE, Q_BASE)
print(f"node2vec p={P_BASE}, q={Q_BASE}: {seg:.0f} s, matriz {Z_base.shape}")

node2vec p=1.0, q=0.5: 30 s, matriz (3489, 16)


## 4. Tarea 1 — Agrupar con los embeddings y comparar con las comunidades
Se agrupa con **K-Means** (K = número de comunidades grandes) y se mide con tres números, todos **comparados contra las comunidades de Louvain** (notebook 8), que se calcularon **sin usar ningún embedding**:
- **ARI** (índice de Rand ajustado): 1 = mismos grupos que Louvain; cerca de 0 = no mejor que el azar.
- **Pureza de vecinos:** para cada proveedor, de sus 10 vecinos más cercanos en el espacio del embedding, qué **fracción está en su misma comunidad**. La línea base es el azar (dos proveedores cualquiera en la misma comunidad).
- **Silueta:** qué tan separados quedan los grupos de K-Means (−1 a 1; más alto es mejor).

Se calcula solo sobre los proveedores de las comunidades grandes (las pequeñas tienen muy pocos miembros en el núcleo).

In [5]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.neighbors import NearestNeighbors

y = etq_com[en_grande]
dep = perfil.reindex(nodos).departamento.values
base_azar = float(((pd.Series(y).value_counts(normalize=True)) ** 2).sum())

def evaluar(Z, nombre, p=None, q=None):
    Zg = Z[en_grande]
    lab = KMeans(K, n_init=10, random_state=SEMILLA).fit_predict(Zg)
    nn = NearestNeighbors(n_neighbors=11, metric="cosine").fit(Zg)
    _, ind = nn.kneighbors(Zg)
    pureza = float((y[ind[:, 1:]] == y[:, None]).mean())
    return {"embedding": nombre, "p": p, "q": q,
            "ARI_vs_louvain": round(adjusted_rand_score(y, lab), 3),
            "pureza_vecinos": round(pureza, 3),
            "silueta": round(float(silhouette_score(Zg, lab, metric="cosine")), 3),
            "ARI_vs_departamento": round(adjusted_rand_score(pd.Series(dep[en_grande]).fillna("s/d"), lab), 3)}, lab

fila, lab_svd = evaluar(Z_svd, "SVD")
metricas = [fila]
fila, lab_base = evaluar(Z_base, f"node2vec (p={P_BASE}, q={Q_BASE})", P_BASE, Q_BASE)
metricas.append(fila)
print(f"línea base de pureza por azar: {base_azar:.3f}")
pd.DataFrame(metricas)

línea base de pureza por azar: 0.120


,embedding,p,q,ARI_vs_louvain,pureza_vecinos,silueta,ARI_vs_departamento
0,SVD,NaN,NaN,0.786,0.914,0.628,0.019
1,"node2vec (p=1.0, q=0.5)",1.0,0.5,0.736,0.931,0.430,0.039


## 5. Efecto de p y q
Se repite node2vec con las 9 combinaciones de `p` y `q` de la grilla y se miden las mismas métricas. En la práctica, `p` y `q` se ajustan con *grid search* (como en la clase).

In [6]:
Z_por_config = {"svd": Z_svd}
Zs = {}
filas = [metricas[0]]
for p in GRILLA_P:
    for q in GRILLA_Q:
        Z, seg = node2vec(p, q)
        Zs[(p, q)] = Z
        f, _ = evaluar(Z, f"node2vec (p={p}, q={q})", p, q)
        filas.append(f)
        print(f"p={p}, q={q}: {seg:.0f} s | ARI {f['ARI_vs_louvain']} | pureza {f['pureza_vecinos']}")
tabla_pq = pd.DataFrame(filas)
tabla_pq

p=0.5, q=0.5: 29 s | ARI 0.726 | pureza 0.929


p=0.5, q=1.0: 30 s | ARI 0.676 | pureza 0.929


p=0.5, q=2.0: 31 s | ARI 0.73 | pureza 0.932


p=1.0, q=0.5: 27 s | ARI 0.736 | pureza 0.931


p=1.0, q=1.0: 25 s | ARI 0.75 | pureza 0.93


p=1.0, q=2.0: 25 s | ARI 0.816 | pureza 0.931


p=2.0, q=0.5: 28 s | ARI 0.733 | pureza 0.929


p=2.0, q=1.0: 26 s | ARI 0.639 | pureza 0.929


p=2.0, q=2.0: 25 s | ARI 0.728 | pureza 0.931


,embedding,p,q,ARI_vs_louvain,pureza_vecinos,silueta,ARI_vs_departamento
0,SVD,NaN,NaN,0.786,0.914,0.628,0.019
1,"node2vec (p=0.5, q=0.5)",0.5,0.5,0.726,0.929,0.426,0.040
2,"node2vec (p=0.5, q=1.0)",0.5,1.0,0.676,0.929,0.423,0.059
3,"node2vec (p=0.5, q=2.0)",0.5,2.0,0.730,0.932,0.428,0.044
4,"node2vec (p=1.0, q=0.5)",1.0,0.5,0.736,0.931,0.430,0.039
5,"node2vec (p=1.0, q=1.0)",1.0,1.0,0.750,0.930,0.428,0.040
6,"node2vec (p=1.0, q=2.0)",1.0,2.0,0.816,0.931,0.425,0.033
7,"node2vec (p=2.0, q=0.5)",2.0,0.5,0.733,0.929,0.425,0.038
8,"node2vec (p=2.0, q=1.0)",2.0,1.0,0.639,0.929,0.378,0.067
9,"node2vec (p=2.0, q=2.0)",2.0,2.0,0.728,0.931,0.431,0.044


### Lectura
- **Los dos embeddings recuperan las comunidades de Louvain:** de los 10 vecinos más cercanos de un proveedor, cerca del 91% (SVD) y del 93% (node2vec) están en su misma comunidad, frente a 12% que daría el azar.
- **No hay un ganador claro.** node2vec tiene un poco más de pureza de vecinos; SVD tiene mayor ARI (0.79 frente a 0.64–0.82 de node2vec) y mejor silueta.
- **p y q casi no cambian la pureza** (0.929 a 0.932). El ARI sí varía (0.64 a 0.82), pero **sin un patrón claro**: K-Means con 12 grupos es sensible y se probó una sola semilla de caminatas. **No se puede afirmar** que un q bajo favorezca a las comunidades, como sugiere la teoría; aquí el núcleo es muy denso (grado medio 188), y en una red así las caminatas se parecen entre sí sin importar p y q.
- **Región:** los grupos de K-Means casi no coinciden con el departamento (ARI cercano a 0.02–0.07). Los embeddings capturan **con quién se cruza** el proveedor, que depende más del rubro que de la región (lo mismo que se vio en las comunidades).
- Que los vecinos del embedding estén en la misma comunidad **no es casualidad**: ambos métodos miran la misma red. Lo útil es poder **visualizar y comparar proveedores** sin volver a calcular comunidades.

## 6. Tarea 2 — Visualizar con t-SNE
**t-SNE** lleva los vectores de 16 números a **2 dimensiones** conservando la cercanía: proveedores cercanos en el mapa tienen vectores parecidos. Se calcula para SVD y para cada configuración de node2vec; la página web deja elegir `p` y `q`.

**Cómo leer el mapa:** la distancia entre **grupos lejanos** no es exacta (t-SNE solo cuida los vecinos cercanos); mire si los colores (comunidades) **forman manchas separadas**.

In [7]:
from sklearn.manifold import TSNE

def tsne(Z):
    return TSNE(n_components=2, perplexity=30, init="pca", random_state=SEMILLA).fit_transform(Z)

filas = []
t0 = time.time()
XY = {"svd": tsne(Z_svd)}
for (p, q), Z in Zs.items():
    XY[f"n2v_p{p}_q{q}"] = tsne(Z)
print(f"t-SNE de {len(XY)} configuraciones: {time.time()-t0:.0f} s")
for conf, xy in XY.items():
    for i in range(n):
        filas.append((conf, nodos[i], round(float(xy[i, 0]), 2), round(float(xy[i, 1]), 2)))
tsne_df = pd.DataFrame(filas, columns=["config", "proveedor_idx", "x", "y"])
tsne_df.groupby("config").size()

t-SNE de 10 configuraciones: 84 s


config
n2v_p0.5_q0.5    3489
n2v_p0.5_q1.0    3489
n2v_p0.5_q2.0    3489
n2v_p1.0_q0.5    3489
n2v_p1.0_q1.0    3489
n2v_p1.0_q2.0    3489
n2v_p2.0_q0.5    3489
n2v_p2.0_q1.0    3489
n2v_p2.0_q2.0    3489
svd              3489
dtype: int64

## 7. Qué estructura capturan
Tres preguntas:
1. ¿**SVD** captura sobre todo **qué tan conectado** está el proveedor (su tamaño)?
2. ¿Los **vecinos** de un proveedor en el embedding se parecen a él en comunidad, región y rubro?
3. ¿Los grupos de node2vec se parecen a los **clústeres por características** de la Parte II (que no miran la red)?

In [8]:
# 1) Relación de cada componente con el grado (log)
lg = np.log1p(grado)
corr_svd = max(abs(np.corrcoef(Z_svd[:, j], lg)[0, 1]) for j in range(D))
corr_n2v = max(abs(np.corrcoef(Zs[(P_BASE, Q_BASE)][:, j], lg)[0, 1]) for j in range(D))
sin_norm = TruncatedSVD(n_components=D, random_state=SEMILLA).fit_transform(Aw)
corr_svd_sin_norm = max(abs(np.corrcoef(sin_norm[:, j], lg)[0, 1]) for j in range(D))
print(f"máxima correlación (en valor absoluto) de un componente con el grado: SVD sin normalizar {corr_svd_sin_norm:.2f} | SVD normalizado {corr_svd:.2f} | node2vec {corr_n2v:.2f}")

# 2) Vecinos de algunos proveedores muy activos
Zn = Zs[(P_BASE, Q_BASE)]
def vecinos_de(i, Z, k=5):
    sim = Z @ Z[i]
    sim[i] = -1
    return np.argsort(-sim)[:k]
atrib_n = perfil.reindex(nodos)
ejemplos = [i for i in np.argsort(-feat.reindex(nodos).n_participaciones.values) if str(feat.loc[nodos[i], "ruc"]).startswith("20")][:6]
filas = []
for i in ejemplos:
    for nombre_emb, Z in [("SVD", Z_svd), (f"node2vec", Zn)]:
        vec = vecinos_de(i, Z)
        filas.append({"proveedor_idx": nodos[i], "embedding": nombre_emb,
                      "vecinos_en_su_comunidad": int((etq_com[vec] == etq_com[i]).sum()),
                      "vecinos_mismo_departamento": int((atrib_n.departamento.values[vec] == atrib_n.departamento.values[i]).sum()),
                      "vecinos_misma_categoria": int((atrib_n.categoria.values[vec] == atrib_n.categoria.values[i]).sum()),
                      "vecinos_idx": [nodos[j] for j in vec]})
vec_df = pd.DataFrame(filas)
print("De los 5 vecinos más cercanos, cuántos coinciden con el proveedor (promedio de los 6 ejemplos):")
print(vec_df.groupby("embedding")[["vecinos_en_su_comunidad", "vecinos_mismo_departamento", "vecinos_misma_categoria"]].mean().round(2))

# 3) Comparación con los clústeres por características de la Parte II.
# Se repite el mismo cálculo de P2_03 (mismas columnas, log, estandarizar, PCA al 90%, K-Means con k = 5, semilla 42)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
NUM = ["n_participaciones", "tasa_exito", "n_adjudicaciones", "monto_total", "monto_mediano", "n_entidades", "n_departamentos", "concentracion", "meses_activo"]
LOG = ["n_participaciones", "n_adjudicaciones", "monto_total", "monto_mediano", "n_entidades", "n_departamentos", "meses_activo"]
fp = pd.read_parquet(f"{RUTA_P2}/features_proveedor").query("n_participaciones >= 5").reset_index(drop=True)
X = fp[NUM].copy(); X[LOG] = np.log1p(X[LOG])
P3 = PCA(n_components=0.90, random_state=42).fit_transform(StandardScaler().fit_transform(X))
fp["tipo_p3"] = KMeans(n_clusters=5, init="k-means++", n_init=10, random_state=42).fit_predict(P3)
tipo = fp.set_index("proveedor_idx").tipo_p3.reindex(nodos).values
ok = ~pd.isna(tipo)
ari_p3 = {}
for nombre_emb, Z in [("SVD", Z_svd), ("node2vec", Zn)]:
    lab_all = KMeans(K, n_init=10, random_state=SEMILLA).fit_predict(Z)
    ari_p3[nombre_emb] = round(float(adjusted_rand_score(tipo[ok], lab_all[ok])), 3)
print(f"ARI entre los grupos del embedding y los tipos de proveedor de la Parte II (k = 5): {ari_p3}  ({int(ok.sum()):,} proveedores del núcleo)")

# ¿Cada comunidad mezcla tipos de la Parte II, o se parece a uno solo?
t3 = pd.crosstab(pd.Series(etq_com[ok]).where(lambda s_: s_.isin(grandes)), pd.Series(tipo[ok].astype(int)), normalize="index").round(2)
t3.index.name = "comunidad"; t3.columns = [f"tipo {c}" for c in t3.columns]
print("Parte II: qué parte de cada comunidad grande pertenece a cada tipo de proveedor (k = 5)")
t3

máxima correlación (en valor absoluto) de un componente con el grado: SVD sin normalizar 0.79 | SVD normalizado 0.38 | node2vec 0.23
De los 5 vecinos más cercanos, cuántos coinciden con el proveedor (promedio de los 6 ejemplos):
           vecinos_en_su_comunidad  vecinos_mismo_departamento  \
embedding                                                        
SVD                           4.67                        4.33   
node2vec                      4.83                        4.50   

           vecinos_misma_categoria  
embedding                           
SVD                           4.50  
node2vec                      4.67  


ARI entre los grupos del embedding y los tipos de proveedor de la Parte II (k = 5): {'SVD': -0.005, 'node2vec': -0.003}  (3,489 proveedores del núcleo)
Parte II: qué parte de cada comunidad grande pertenece a cada tipo de proveedor (k = 5)


,tipo 0,tipo 1,tipo 2,tipo 3
comunidad,,,,
0.0,0.38,0.20,0.41,0.0
1.0,0.09,0.10,0.81,0.0
2.0,0.30,0.18,0.52,0.0
3.0,0.26,0.16,0.57,0.0
4.0,0.15,0.17,0.68,0.0
5.0,0.18,0.10,0.73,0.0
6.0,0.12,0.12,0.77,0.0
7.0,0.03,0.04,0.94,0.0
8.0,0.04,0.08,0.88,0.0


### Lectura
- **SVD sin normalizar sí refleja el tamaño:** un componente llega a una correlación de 0.79 con el número de conexiones (0.38 al normalizar los vectores y 0.23 con node2vec). Es el efecto de «la dirección dominante» de la matriz, el mismo que se vio con HITS en la Parte IV.
- **Los vecinos de ambos embeddings se parecen al proveedor** en comunidad, región y rubro: en los 6 proveedores más activos, 4.7 (SVD) y 4.8 (node2vec) de los 5 vecinos más cercanos están en su misma comunidad. El vector captura **con quién se cruza** el proveedor.
- **Relación con los tipos de la Parte II (que no miran la red):** el ARI es **cercano a 0** (−0.005 y −0.003): los grupos de los embeddings **no coinciden** con los tipos por características. Además, el núcleo está formado casi solo por proveedores del tipo 2 (muy activos: mediana de 99 participaciones); dentro de cada comunidad grande el tipo 2 reúne entre 41% y 94%. La red y las características son **dos miradas distintas**: los tipos agrupan por **cómo se comporta** el proveedor (cuánto gana, con cuántas entidades) y los embeddings por **con quién se cruza**.
- Todo esto describe **similitud de posición en la red**; no dice nada sobre la conducta de ningún proveedor.

## 8. Archivos para la página web
**Datos personales:** el RUC que no empieza con 20 se enmascara y el nombre de una persona natural se reemplaza por «Persona natural».

In [9]:
def enmascarar(ruc):
    r = str(ruc); return r if r.startswith("20") else r[:5] + "****" + r[-2:]
def nombre_visible(ruc, nombre):
    r = str(ruc); return f"Persona natural (RUC {enmascarar(r)})" if r[:2] in ("10", "15", "16", "17") else nombre

orden_com = pd.Series(etq_com).value_counts().index.tolist()
grupo = {c: (f"Comunidad {c}" if c in grandes else "Otras comunidades") for c in orden_com}
nodos_web = pd.DataFrame({
    "proveedor_idx": nodos,
    "nombre": [nombre_visible(feat.loc[x, "ruc"], feat.loc[x, "nombre"]) for x in nodos],
    "ruc": [enmascarar(feat.loc[x, "ruc"]) for x in nodos],
    "comunidad": [grupo[c] for c in etq_com],
    "departamento": atrib_n.departamento.str.title().values,
    "categoria": atrib_n.categoria.values,
    "conexiones": grado,
    "participaciones": feat.reindex(nodos).n_participaciones.values,
})
guardar_csv("emb_nodos.csv", nodos_web)
guardar_csv("emb_tsne.csv", tsne_df)
guardar_csv("emb_metricas.csv", tabla_pq)

# vecinos de ejemplo (con nombres públicos)
vv = []
for _, fila in vec_df.iterrows():
    for rango, j in enumerate(fila.vecinos_idx, 1):
        vv.append({"proveedor": nombre_visible(feat.loc[fila.proveedor_idx, "ruc"], feat.loc[fila.proveedor_idx, "nombre"]),
                   "embedding": fila.embedding, "rango": rango,
                   "vecino": nombre_visible(feat.loc[j, "ruc"], feat.loc[j, "nombre"]),
                   "misma_comunidad": bool(com.get(j) == com.get(fila.proveedor_idx))})
guardar_csv("emb_vecinos.csv", pd.DataFrame(vv))

guardar_json("p6_resumen.json", {
    "nucleo": {"descripcion": f"Proveedores con ≥ {MIN_PART_NUCLEO} participaciones, aristas con peso ≥ 2, componente mayor.",
               "nodos": n, "aristas": G.number_of_edges(), "grado_medio": round(float(grado.mean()), 1)},
    "parametros": {"D": D, "num_walks": NUM_WALKS, "largo": LARGO, "ventana": VENTANA, "p_base": P_BASE, "q_base": Q_BASE,
                   "K": K, "min_grande": MIN_GRANDE, "proveedores_en_comunidades_grandes": int(en_grande.sum())},
    "linea_base_pureza_azar": round(base_azar, 3),
    "varianza_svd_16": round(float(svd.explained_variance_ratio_.sum()), 3),
    "correlacion_con_grado": {"svd_sin_normalizar": round(corr_svd_sin_norm, 2), "svd_normalizado": round(corr_svd, 2), "node2vec": round(corr_n2v, 2)},
    "ari_vs_tipos_parte2": ari_p3,
})

guardado: emb_nodos.csv 3489 filas


guardado: emb_tsne.csv 34890 filas
guardado: emb_metricas.csv 10 filas
guardado: emb_vecinos.csv 60 filas
guardado: p6_resumen.json
